# 01r · 모듈 01 복습 (닫힌 책): 이산시간 마르코프 사슬 (Module 01 Review (Closed Book): Discrete-Time Markov Chains)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e Ch. 1 (§1.1–1.10) — 복습 범위 전체, 특히 §1.3, §1.4, §1.8, §1.10 · Ross 12e Ch. 4 (§4.1–4.9) — §4.5.1(파산), §4.7(분기), §4.8–4.9(가역·MCMC); 4장 끝 연습문제로 보충 · Norris Ch. 1 — 증명을 다시 읽고 싶을 때 §1.5(재귀), §1.7(불변분포), §1.8(수렴), §1.9(시간 반전) |
| 선수 노트북 | 01a–01g (정의 → 분류 → 흡수 → 정상분포 → 수렴 → 가역·MCMC → 분기과정) |
| 예상 소요 | 60분 |
| 상태 | draft |

이 노트북은 새 정의를 도입하지 않는다. 목적은 세 가지다.

1. 모듈 01의 핵심 정의·정리·숫자 12문항을 **책과 이전 노트북을 보지 않고** 재구성하고 자기채점한다.
2. 세 가지 핵심 유도 — 도박꾼 파산(01c), 정상분포(01d), 소멸확률(01g) — 를 **백지에서** 다시 수행하고, 코드가 공식에서 계산한 정확값과 시뮬레이션으로 채점한다.
3. 자기채점표로 약한 노트북을 찾아 `log/progress.md`에 기록하고, 모듈 01의 함정 목록을 한 장으로 정리한다.

규칙: 1절과 3절의 ✍️ 칸은 책과 이전 노트북을 **닫고** 채웁니다. 정답(`<details>`)은 채운 뒤에만 여세요. 코드 셀은 유도가 끝난 뒤 실행해 "내 손 계산이 맞았는지"를 기계로 확인하는 용도입니다.

모듈 01을 관통하는 질문은 하나다 — **"이 사슬은 오랜 시간 뒤 어떻게 보이는가?"** — 흡수(01c: $B,t$), 평형(01d: $\pi$), 수렴 속도(01e: $\lambda_*$), 설계(01f: MH), 무한 상태(01g/01x: 소멸·재귀).

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (simulate_chain, hitting_times, absorption, stationary, stationary_eig,
                          stationary_power, gamblers_ruin_matrix, gamblers_ruin_prob,
                          gamblers_ruin_duration, galton_watson, extinction_probability)

SEED, rng = rng_for("01r")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. 닫힌 책 — 12문항 (Closed-Book Recall)

각 문항의 답을 종이에 먼저 적으세요. 한 문항당 2–3분을 넘기지 말고, 막히면 0점으로 두고 넘어가세요 — 어디가 약한지 아는 것이 이 절의 목적입니다. 다 적은 뒤 정답을 펼쳐 5절의 자기채점표에 0/1/2점을 기록합니다. 괄호 안은 원 노트북 id입니다. 정답의 숫자 중 Q6(파산)과 Q12(소멸확률)은 3절에서 코드가 다시 계산하고, 나머지 숫자는 정답 끝에 적힌 원 노트북의 절에서 확인하세요.

**Q1.** (01a) 마르코프 성질과 시간 동질성을 식으로 쓰고, 채프먼-콜모고로프(Chapman-Kolmogorov) 방정식이 $P^{(n)}=P^n$을 뜻하는 이유를 한 줄로 말하시오.

<details><summary>정답</summary>

$P(X_{n+1}=y\mid X_n=x,\ \text{과거})=P(X_{n+1}=y\mid X_n=x)=p(x,y)$ — 조건부 확률이 과거와 무관(마르코프)하고 $n$과 무관(시간 동질). 채프먼-콜모고로프 $p^{m+n}(x,y)=\sum_z p^m(x,z)\,p^n(z,y)$는 행렬 곱의 정의 그 자체이므로 $P^{(m+n)}=P^{(m)}P^{(n)}$, 귀납적으로 $P^{(n)}=P^n$. (01a 정리 B)

</details>

**Q2.** (01a) 날씨 사슬 $P=\begin{pmatrix}0.7&0.3\\0.4&0.6\end{pmatrix}$의 $p^2(0,0)$, 정상분포, 두 번째 고유값은?

<details><summary>정답</summary>

$p^2(0,0)=0.7^2+0.3\cdot0.4=0.61$; $\pi=(4/7,\,3/7)$ ($\pi_0\cdot0.3=\pi_1\cdot0.4$); $\lambda_2=1-0.3-0.4=0.3$ (2상태 사슬은 $\lambda_2=1-p(0,1)-p(1,0)$). (01a §4.1, 01e §4.1)

</details>

**Q3.** (01b) 재귀/일시의 정의(귀환확률로)와 재귀 기준(급수로)은? 일시 상태의 방문 횟수 분포는?

<details><summary>정답</summary>

$\rho_{xx}=P_x(T_x<\infty)$, $T_x=\min\{n\ge1:X_n=x\}$. 재귀 $\iff\rho_{xx}=1\iff\sum_n p^n(x,x)=\infty$ (기대 방문 횟수 $=\sum_np^n(x,x)=1/(1-\rho_{xx})$). 일시 상태의 방문 횟수(출발 포함)는 성공확률 $1-\rho_{xx}$인 기하분포, 평균 $1/(1-\rho_{xx})$. (01b 정리 1)

</details>

**Q4.** (01b) 유한 사슬에서 닫힌 클래스와 새는 클래스는 각각 무엇인가? 주기의 정의는? 자기 루프가 있으면 주기는?

<details><summary>정답</summary>

유한 사슬: 닫힌 클래스 = 재귀, 닫히지 않은(새는) 클래스 = 일시. 주기 $d(x)=\gcd\{n\ge1:p^n(x,x)>0\}$ — 클래스 성질이다. $p(x,x)>0$이면 $1\in\{n:p^n(x,x)>0\}$이므로 $d(x)=1$. (01b 정리 3, 4)

</details>

**Q5.** (01c) 기본행렬 $N$의 정의와 의미, 흡수확률 $B$와 기대 흡수시간 $t$의 공식은?

<details><summary>정답</summary>

$P=\begin{pmatrix}Q&R\\0&I\end{pmatrix}$로 일시/흡수 상태를 나누면 $N=(I-Q)^{-1}=\sum_{k\ge0}Q^k$, $N_{ij}$ = $i$에서 출발해 흡수 전 $j$를 방문하는 횟수의 기댓값. $B=NR$ ($B_{ia}$ = $a$에 흡수될 확률), $t=N\mathbf 1$ (기대 흡수시간). (01c 정리 1, 2)

</details>

**Q6.** (01c) 도박꾼 파산 $N=10$, $p=0.45$, $i=5$: 승리 확률과 기대 지속시간의 공식과 값은? $p=1/2$이면?

<details><summary>정답</summary>

$r=q/p=11/9$: $h(i)=\dfrac{1-r^i}{1-r^N}=0.2683$; $\tau(i)=\dfrac{i}{q-p}-\dfrac{N}{q-p}h(i)=50-100\cdot0.2683=23.17$. $p=1/2$: $h=i/N=0.5$, $\tau=i(N-i)=25$. (01c 정리 3; R1에서 재유도)

</details>

**Q7.** (01d) 정상분포의 정의와 세 계산법, 그리고 $\pi(x)$와 평균 귀환시간의 관계는? Ehrenfest($n$)의 정상분포는?

<details><summary>정답</summary>

$\pi P=\pi$, $\sum_x\pi(x)=1$ (흐름 균형 $\sum_x\pi(x)p(x,y)=\pi(y)$). 계산: (i) 선형계(방정식 하나를 정규화로 대체), (ii) $P^\top$의 고유값 1 고유벡터, (iii) $P^n$의 행 / 장기 시간 빈도. 기약 유한이면 $\pi(x)=1/E_xT_x$. Ehrenfest($n$): $\pi=\text{Bin}(n,1/2)$. (01d 정리 1, 2, 4)

</details>

**Q8.** (01d/01e) 정상분포가 유일해도 $P^n$이 수렴하지 않는 예와, 그때 수렴하는 양은? 수렴 정리의 가정은?

<details><summary>정답</summary>

$\{0,\dots,4\}$ 위 반사 랜덤 워크(주기 2): $\pi$는 유일하지만 $P^{1000}[0,:]=(1/4,0,1/2,0,1/4)$처럼 짝·홀 상태를 번갈아 밟는다. 그래도 세자로(Cesàro) 평균 $\frac1n\sum_{k<n}P^k\to\mathbf 1\pi$와 시간 평균(에르고딕 정리)은 수렴한다. 수렴 정리 $P^n(x,\cdot)\to\pi$의 가정: 기약 + 비주기 (+ 유한 또는 양재귀). (01d §4.4, 01e 정리 2, 4)

</details>

**Q9.** (01e) TV 거리의 두 정의, 결합 부등식, 가역 사슬의 스펙트럼 상계는?

<details><summary>정답</summary>

$\|\mu-\nu\|_{TV}=\max_A|\mu(A)-\nu(A)|=\frac12\sum_x|\mu(x)-\nu(x)|$. 결합(coupling) 부등식: $\|P^n(x,\cdot)-\pi\|_{TV}\le P(T_c>n)$, $T_c$ = 두 사슬이 만나는 시각. 가역 사슬: $\|P^n(x,\cdot)-\pi\|_{TV}\le\dfrac{\lambda_*^n}{2\sqrt{\pi(x)}}$, $\lambda_*=\max(|\lambda_2|,|\lambda_{\min}|)$. (01e 정리 1, 3)

</details>

**Q10.** (01f) 세부균형 조건과 그것이 정상성을 함의하는 한 줄 증명, Metropolis-Hastings 수락확률의 일반식은?

<details><summary>정답</summary>

세부균형 $\pi(x)p(x,y)=\pi(y)p(y,x)$ (모든 $x,y$). 증명: $\sum_x\pi(x)p(x,y)=\sum_x\pi(y)p(y,x)=\pi(y)\sum_xp(y,x)=\pi(y)$. MH: 제안 $q(x,y)$, 수락 $\alpha(x,y)=\min\!\Big(1,\dfrac{\pi(y)q(y,x)}{\pi(x)q(x,y)}\Big)$ — 정규화 상수 없이 계산된다. (01f 정리 1, 2)

</details>

**Q11.** (01f) MCMC 표본 $n$개의 평균의 분산은 대략 무엇이고, 가역 사슬에서 $\tau$의 상계는? 수락률이 높으면 좋은 사슬인가?

<details><summary>정답</summary>

$\operatorname{Var}(\bar f_n)\approx\sigma_f^2\tau/n$, $\tau=1+2\sum_{k\ge1}\rho_k$ (적분 자기상관 시간), 가역이면 $\tau\le\dfrac{1+\lambda_2}{1-\lambda_2}$. 아니다: 01f의 $\pm1$ 제안은 수락률 81%지만 $\tau\approx20$, 균등 독립 제안은 수락률 27%지만 $\lambda_2=0.75$로 훨씬 빨리 섞인다. 효율의 척도는 $\tau$($\lambda_2$)이지 수락률이 아니다. (01f 정리 4, E2)

</details>

**Q12.** (01g) 소멸확률의 특징(pgf로), $m$에 따른 분류, 자손분포 $(0.25,0.25,0.5)$의 $q$, 임계 과정의 생존확률 감쇠율은?

<details><summary>정답</summary>

$q$ = $\varphi(s)=s$의 $[0,1]$에서 **최소** 비음 근 $=\lim_n\varphi_n(0)$ ($\varphi_n=\varphi\circ\varphi_{n-1}$). $m=\varphi'(1)\le1$ ($p_1<1$)이면 $q=1$, $m>1$이면 $q<1$. $(0.25,0.25,0.5)$: $m=1.25$, $q=1/2$. 임계($m=1$, $\sigma^2<\infty$): $P(Z_n>0)\sim 2/(\sigma^2n)$. (01g 정리 2, 3; R3에서 재유도)

</details>

## 2. 모듈 01 함정 목록 (한 장) — 채점이 끝난 뒤에 펼치기

1절을 채점한 뒤에만 여세요. 3절의 재유도에서 코드 검사가 실패하면 여기로 돌아옵니다. 각 항목의 출처는 원 노트북의 "함정" 절입니다.

<details><summary>함정 10개 (모듈 01)</summary>

1. **행렬 거듭제곱** (01a): `P**n`은 원소별 거듭제곱, $P^n$은 `np.linalg.matrix_power(P, n)`. 분포는 행벡터 `mu @ P`로 밀지 `P @ mu`가 아니다.
2. **주기 = gcd** (01b): 주기 $d$는 "매 $d$걸음마다 돌아온다"가 아니라 "귀환 가능 시각들의 최대공약수"다. 자기 루프 하나면 1.
3. **지평 절단** (01c): 고정 걸음 수로 자르면 미흡수 경로가 $E[T]$를 과소추정한다. 미흡수 비율을 항상 출력한다.
4. **특이 $I-Q$** (01c): 일시 집합 $T$에 재귀 상태를 섞으면 $Q^n\not\to0$이라 $I-Q$가 특이해진다.
5. **정상분포 존재 ≠ $P^n$ 수렴** (01d/01e): 주기적 사슬은 $\pi$가 유일해도 $P^n$이 진동한다. 세자로 평균만 수렴.
6. **경험 TV 거리의 양의 편향** (01e): 히스토그램과 $\pi$의 TV 거리는 표본 잡음 때문에 항상 양수다. MC 검증은 단일 확률 $P(X_n=x)$로 한다.
7. **$\lambda_*$는 $|\lambda_{\min}|$도 본다** (01e): 비게으른 Ehrenfest(20)는 $\lambda_2=0.9$이지만 $\lambda_{\min}=-1$이라 수렴하지 않는다.
8. **MCMC SE** (01f): `std/sqrt(n)`은 $\sqrt\tau$배 과소평가. 배치 평균이나 독립 사슬로 SE를 구한다. 수락률은 효율 척도가 아니다.
9. **$\varphi_n(0)$ vs $q$** (01g): $n$세대까지의 소멸 비율은 $\varphi_n(0)<q$이지 $q$가 아니다. 유한 $n$의 MC를 $q$와 비교하면 어긋난다.
10. **평균 vs 전형** (01g): $EZ_n=m^n\to\infty$여도 절반이 소멸할 수 있다 — 평균은 소수의 폭발한 경로가 끌어올린다.

</details>

## 3. 재유도 — 세 가지 핵심 결과 (Re-derivations)

세 결과를 각각 **백지에서** 유도하세요. 공식을 기억해 적는 것이 아니라 1단계 분석 / 흐름 균형 / 첫 세대 조건화 논법을 처음부터 다시 쓰는 것입니다.

셋은 모두 "첫 걸음(첫 세대)에 조건화"라는 하나의 기법이다. 각 결과마다 (i) 명제와 "가정이 왜 필요한가", (ii) ✍️ 내 유도 칸, (iii) 기대 답안(`<details>`), (iv) 코드 채점이 이어진다. 코드 채점은 두 층이다: **정확식 vs 정확식**(`allclose`: 내가 유도한 공식 vs `spkit`의 행렬 계산)과 **MC vs 정확식**(4 SE). 둘 중 하나만 통과하면 어느 쪽이 틀렸는지 자기채점표 메모에 적는다.

### R1 재유도: 도박꾼 파산 (Gambler's Ruin)

**명제.** 자본 $i\in\{0,\dots,N\}$, 매 판 $+1$ 확률 $p$, $-1$ 확률 $q=1-p$, $0$과 $N$에서 흡수. $h(i)=P_i(\text{reach }N\text{ before }0)$, $\tau(i)=E_i[T]$ ($T$ = 흡수 시각)에 대해

$$h(i)=\frac{1-r^i}{1-r^N}\ (r=q/p\ne1),\qquad h(i)=\frac iN\ (p=\tfrac12);\qquad \tau(i)=\frac{i}{q-p}-\frac{N}{q-p}h(i)\ (p\ne\tfrac12),\qquad \tau(i)=i(N-i)\ (p=\tfrac12).$$

(a) 1단계 분석으로 $h$의 차분방정식을 세우고 두 경우의 닫힌 식을 유도하라. (b) $\tau$의 차분방정식을 세우고 특수해 + 동차해로 풀어라. (c) $N=10$, $p=0.45$, $i=5$의 값을 손으로 계산하라.

**가정이 왜 필요한가.**
- 마르코프 성질 + 시간 동질: 첫 걸음 뒤의 사슬이 새 출발점에서의 같은 사슬이어야 $h(i)=p\,h(i+1)+q\,h(i-1)$이 성립한다.
- $T<\infty$ a.s. (일시 상태 $1,\dots,N-1$에서 반드시 흡수): 그래야 $h(i)+P_i(\text{reach }0)=1$이고 $\tau(i)<\infty$라 차분방정식을 $\tau$에 대해 쓸 수 있다. 행렬 쪽에서는 $Q^n\to0$ ⇒ $I-Q$ 가역.

✍️ 내 유도 (R1): (책을 덮고 여기에 (a)(b)(c)를 적으세요)

In [ ]:
# 여기에 풀이: 손으로 계산한 h(5), tau(5) (p = 0.45) 를 적어 두고 아래 코드 채점과 비교

<details><summary>기대 답안 (R1)</summary>

**(a) $h$.** 첫 걸음에 조건화: $h(i)=p\,h(i+1)+q\,h(i-1)$, $h(0)=0$, $h(N)=1$. $h(i)=(p+q)h(i)$이므로 $p\,(h(i+1)-h(i))=q\,(h(i)-h(i-1))$, 즉 증분 $\Delta_i=h(i+1)-h(i)$가 $\Delta_i=r\Delta_{i-1}=r^i\Delta_0$ ($r=q/p$). 망원합 $h(i)=\sum_{k<i}\Delta_k=h(1)\dfrac{1-r^i}{1-r}$ ($r\ne1$), $h(N)=1$로 $h(1)$을 결정하면 $h(i)=\dfrac{1-r^i}{1-r^N}$. $p=\frac12$ ($r=1$): 증분이 모두 같으므로 $h$는 선형, $h(i)=i/N$.

**(b) $\tau$.** 첫 걸음에 1이 들고 조건화: $\tau(i)=1+p\,\tau(i+1)+q\,\tau(i-1)$, $\tau(0)=\tau(N)=0$. 비동차 선형 차분방정식. 특수해로 $\tau=c\,i$를 넣으면 $ci=1+ci+c(p-q)$ ⇒ $c=1/(q-p)$ ($p\ne\frac12$). 동차해는 (a)와 같은 방정식이므로 $A+Br^i$. 경계: $\tau(0)=0\Rightarrow A=-B$; $\tau(N)=0\Rightarrow \frac{N}{q-p}+B(r^N-1)=0\Rightarrow B=\frac{N}{(q-p)(1-r^N)}$. 정리하면
$$\tau(i)=\frac{i}{q-p}+B(r^i-1)=\frac{i}{q-p}-\frac{N}{q-p}\cdot\frac{1-r^i}{1-r^N}=\frac{i}{q-p}-\frac{N}{q-p}h(i).$$
$p=\frac12$: 특수해 $-i^2$ ($-i^2=1-\frac12(i+1)^2-\frac12(i-1)^2=-i^2$ ✓), 동차해 $A+Bi$; $\tau(0)=0\Rightarrow A=0$, $\tau(N)=0\Rightarrow B=N$: $\tau(i)=i(N-i)$.

**(c) 수치.** $r=0.55/0.45=11/9$, $r^5\approx2.727$, $r^{10}\approx7.439$: $h(5)=(1-2.727)/(1-7.439)=1.727/6.439\approx0.2683$. $q-p=0.1$: $\tau(5)=50-100\cdot0.2683\approx23.17$. 해석: 공정하면 25판이지만 불리하면 빨리 끝난다(대개 파산 쪽으로). (01c 정리 3; Durrett 3e §1.8–§1.9, Ross 12e §4.5.1)

</details>

코드 채점 1 — 정확식 vs 정확식: 유도한 공식을 코드로 그대로 옮긴 값, `spkit`의 닫힌 공식, 그리고 기본행렬 $N=(I-Q)^{-1}$에서 나온 $B[4,1]$, $t[4]$ (상태 $1,\dots,9$ 중 다섯 번째가 $i=5$)를 $10^{-10}$ 수준에서 비교한다.

In [ ]:
N_CAP, P_WIN, I0 = 10, 0.45, 5
r = (1 - P_WIN) / P_WIN
h_derived = (1 - r**I0) / (1 - r**N_CAP)                                 # (a)에서 유도한 식
tau_derived = I0 / (1 - 2 * P_WIN) - N_CAP / (1 - 2 * P_WIN) * h_derived  # (b)에서 유도한 식
h_exact, tau_exact = gamblers_ruin_prob(I0, N_CAP, P_WIN), gamblers_ruin_duration(I0, N_CAP, P_WIN)

P_gr = gamblers_ruin_matrix(N_CAP, P_WIN)
ab = absorption(P_gr, list(range(1, N_CAP)), [0, N_CAP])            # 일시 1..9, 흡수 [0, 10]
B_5, t_5 = ab["B"][I0 - 1, 1], ab["t"][I0 - 1]
print(f"h(5):   derived {h_derived:.10f}, spkit {h_exact:.10f}, N@R {B_5:.10f}")
print(f"tau(5): derived {tau_derived:.10f}, spkit {tau_exact:.10f}, N@1 {t_5:.10f}")
print(f"p = 1/2: h = {gamblers_ruin_prob(I0, N_CAP, 0.5)}, tau = {gamblers_ruin_duration(I0, N_CAP, 0.5)}")
np.testing.assert_allclose([h_derived, B_5], h_exact, rtol=1e-10)
np.testing.assert_allclose([tau_derived, t_5], tau_exact, rtol=1e-10)
print("R1 exact check: ok")

코드 채점 2 — MC vs 정확식: 자본 5에서 출발한 경로 `N_PATHS`개를 300걸음 돌린다. $P_5(T>300)=Q^{300}[4,:]\,\mathbf 1=(Q^{300}\mathbf 1)_4\approx10^{-7}$이라 절단 편향은 SE보다 훨씬 작지만, 함정 3대로 미흡수 경로 수를 출력한다(미흡수 경로가 있으면 $T=300$으로 채워 하한으로 쓴다).

In [ ]:
N_STEPS_GR = 300
paths_gr = simulate_chain(P_gr, I0, N_STEPS_GR, rng, N_PATHS)
T_gr = hitting_times(paths_gr, {0, N_CAP})
n_unabsorbed = int(np.sum(~np.isfinite(T_gr)))
p_trunc = np.linalg.matrix_power(P_gr[1:N_CAP, 1:N_CAP], N_STEPS_GR)[I0 - 1].sum()
p_win_hat = mc_proportion(paths_gr[:, -1] == N_CAP)
dur_hat = mc_estimate(np.where(np.isfinite(T_gr), T_gr, N_STEPS_GR))
print(f"unabsorbed after {N_STEPS_GR} steps: {n_unabsorbed} of {N_PATHS}  (exact P(T > 300) = {p_trunc:.2e})")
print(f"P(win):   {p_win_hat}   exact {h_exact:.6f}")
print(f"E[T]:     {dur_hat}   exact {tau_exact:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for k in range(12):
    ax1.step(np.arange(N_STEPS_GR + 1), paths_gr[k], where="post", lw=0.9, alpha=0.8)
ax1.set(xlim=(0, 80), xlabel="n (rounds)", ylabel="fortune", title=f"Gambler's ruin paths (p = {P_WIN}, start 5)")
ax1.axhline(0, color="k", lw=0.8); ax1.axhline(N_CAP, color="k", lw=0.8)
Tf = T_gr[np.isfinite(T_gr)]
ax2.hist(Tf, bins=np.arange(0, 120, 2), density=True, alpha=0.6, label=f"MC ({N_PATHS} paths)")
ax2.axvline(tau_exact, color="C3", ls="--", label=f"exact E[T] = {tau_exact:.2f}")
ax2.set(xlabel="T (absorption time)", ylabel="density", title="Absorption time distribution"); ax2.legend();

### R2 재유도: 정상분포 (Stationary Distribution)

**명제.** (a) 정상분포의 정의를 흐름 균형(flow balance)으로 쓰고, 사회 이동 사슬

$$P=\begin{pmatrix}0.7&0.2&0.1\\0.3&0.5&0.2\\0.2&0.4&0.4\end{pmatrix}$$

의 $\pi$를 **손으로** 푸시오 (분수로). (b) '기약 유한 사슬에서 $\pi(x)=1/E_xT_x$'의 증명 개요를 **다섯 줄**로 쓰시오 (한 사이클 동안의 방문 횟수 측도). (c) $P$의 두 번째 고유값의 절댓값을 손으로 구하시오 (힌트: 대각합과 행렬식).

**가정이 왜 필요한가.**
- 유한: 정상분포의 **존재**(브라우어르/컴팩트성 또는 $P^n$ 행의 세자로 극한). 무한 사슬은 영재귀·일시일 때 정상분포가 없다(01x).
- 기약: **유일성**. 닫힌 클래스가 둘이면 각 클래스의 정상분포의 볼록결합이 모두 정상이다.
- (b)에서 $E_xT_x<\infty$ (유한 기약 ⇒ 양재귀)가 있어야 측도 $\mu$를 정규화할 수 있다.

✍️ 내 유도 (R2): (책을 덮고 여기에 (a)(b)(c)를 적으세요)

In [ ]:
# 여기에 풀이: 손으로 푼 pi (분수) 와 |lambda_2| 를 적어 두고 아래 코드 채점과 비교

<details><summary>기대 답안 (R2)</summary>

**(a) 흐름 균형.** $\pi(y)=\sum_x\pi(x)p(x,y)$ (모든 $y$), $\sum_y\pi(y)=1$: 정상 상태에서 $y$로 들어오는 확률 흐름 $=\pi(y)$ = $y$에서 나가는 흐름. 세 방정식 중 하나는 나머지의 결과(열합이 자동으로 맞음)이므로 둘만 쓴다.
- 열 0: $\pi_0=0.7\pi_0+0.3\pi_1+0.2\pi_2\Rightarrow 0.3\pi_0=0.3\pi_1+0.2\pi_2$.
- 열 2: $\pi_2=0.1\pi_0+0.2\pi_1+0.4\pi_2\Rightarrow 0.6\pi_2=0.1\pi_0+0.2\pi_1\Rightarrow\pi_0=6\pi_2-2\pi_1$.
- 대입: $0.3(6\pi_2-2\pi_1)=0.3\pi_1+0.2\pi_2\Rightarrow1.6\pi_2=0.9\pi_1\Rightarrow\pi_1=\frac{16}{9}\pi_2$, $\pi_0=6\pi_2-\frac{32}{9}\pi_2=\frac{22}{9}\pi_2$.
- $\pi\propto(22,16,9)$, 합 47: $\pi=\left(\frac{22}{47},\frac{16}{47},\frac{9}{47}\right)\approx(0.4681,0.3404,0.1915)$. (검산: 열 1: $0.2\cdot22+0.5\cdot16+0.4\cdot9=4.4+8+3.6=16$ ✓; $\pi_1=\frac{8}{11}\pi_0$.)

**(b) $\pi(x)=1/E_xT_x$ 증명 개요 (다섯 줄).**
1. 기약 유한 ⇒ 모든 상태 재귀이고 $E_xT_x<\infty$. 고정한 $x$에 대해 $\mu(y)=E_x\big[\sum_{n=0}^{T_x-1}\mathbf 1\{X_n=y\}\big]$ = 한 사이클 동안 $y$를 방문한 기대 횟수라 두자 ($\mu(x)=1$).
2. $\sum_y\mu(y)=E_xT_x<\infty$ (사이클 길이).
3. $\mu P=\mu$: 사이클을 한 칸 밀면 $\sum_{n=1}^{T_x}\mathbf 1\{X_n=y\}$의 기댓값도 $\mu(y)$인데(시각 0의 $x$ 방문과 시각 $T_x$의 $x$ 방문이 맞교환), 강마르코프/조건화로 이것은 $\sum_z\mu(z)p(z,y)$와 같다.
4. 따라서 $\pi=\mu/E_xT_x$는 정상분포이고 $\pi(x)=\mu(x)/E_xT_x=1/E_xT_x$.
5. 기약성으로 정상분포는 유일하므로 이 $\pi$가 **그** 정상분포다 — $x$를 바꿔도 같은 벡터가 나온다. (01d 정리 1; 완전 증명은 04c, Norris §1.7)

**(c) $|\lambda_2|$.** 고유값 $1,\lambda_2,\lambda_3$: 대각합 $1.6=1+\lambda_2+\lambda_3$, 행렬식 $\det P=0.084-0.016+0.002=0.07=\lambda_2\lambda_3$. $\lambda^2-0.6\lambda+0.07=0\Rightarrow\lambda=0.3\pm\sqrt{0.02}=0.4414,\ 0.1586$. 수렴 속도 $|\lambda_2|=0.4414$: 40걸음이면 $0.4414^{40}\approx10^{-14}$이라 $P^{40}$의 행은 기계 정밀도에서 $\pi$다.

</details>

코드 채점 1 — 정확식 vs 정확식: 손으로 푼 $(22,16,9)/47$을 `stationary`(선형계), `stationary_eig`(고유벡터), `stationary_power`(거듭제곱)의 세 계산법과 비교하고, $|\lambda_2|$와 $1/\pi_0$(평균 귀환시간)도 계산한다.

In [ ]:
P_mob = np.array([[0.7, 0.2, 0.1], [0.3, 0.5, 0.2], [0.2, 0.4, 0.4]])   # 사회 이동 사슬 (01d)
pi_hand = np.array([22, 16, 9]) / 47                                     # (a)에서 손으로 푼 값
pi_mob, pi_eig, pi_pow = stationary(P_mob), stationary_eig(P_mob), stationary_power(P_mob, 40)
lam_mob = np.sort(np.abs(np.linalg.eigvals(P_mob)))[::-1]
lam2_hand = 0.3 + np.sqrt(0.02)                                          # (c): 대각합·행렬식에서
ret_exact = 1 / pi_mob[0]                                                # E_0 T_0 = 1/pi(0) = 47/22
print("pi: solve", np.round(pi_mob, 6), " eig", np.round(pi_eig, 6), " P^40 row", np.round(pi_pow, 6))
print(f"|lambda|: {np.round(lam_mob, 6)},  hand |lambda_2| = {lam2_hand:.6f};  E_0 T_0 = 1/pi_0 = {ret_exact:.6f}")
for v in (pi_mob, pi_eig, pi_pow):
    np.testing.assert_allclose(v, pi_hand, atol=1e-12)
np.testing.assert_allclose(lam_mob[1], lam2_hand, rtol=1e-12)
np.testing.assert_allclose(pi_mob @ P_mob, pi_mob, atol=1e-15)          # 흐름 균형 그 자체
print("R2 exact check: ok")

코드 채점 2 — MC vs 정확식: (i) 상태 2(최하층)에서 출발한 경로 `N_PATHS`개의 시각 40 **단면**에서 상태 0의 비율 — 정확값은 $\pi_0$ (= $P^{40}[2,0]$, 차이 $10^{-14}$). (ii) 상태 0에서 출발한 귀환시간 $T_0$의 표본평균 — 정확값 $1/\pi_0=47/22$ (증명 개요 (b)의 내용). 귀환시간은 60걸음에서 자르므로 함정 3대로 미귀환 경로 수와 정확한 꼬리 $P_0(T_0>60)=P[0,\{1,2\}]\,Q_{\{1,2\}}^{59}\,\mathbf 1$을 같이 출력한다($\approx10^{-9}$라 절단 편향은 무시할 만하다). 그림은 $P(X_n=0\mid X_0=2)$가 $|\lambda_2|^n$ 속도로 $\pi_0$에 붙는 모습이다.

In [ ]:
N_STEPS_MOB = 40
paths_mob = simulate_chain(P_mob, 2, N_STEPS_MOB, rng, N_PATHS)
pi0_hat = mc_proportion(paths_mob[:, N_STEPS_MOB] == 0)
paths_0 = simulate_chain(P_mob, 0, 60, rng, N_PATHS)
T_ret = hitting_times(paths_0[:, 1:], 0) + 1                    # 시각 1 이후 첫 귀환
n_unreturned = int(np.sum(~np.isfinite(T_ret)))
p_tail = P_mob[0, 1:] @ np.linalg.matrix_power(P_mob[1:, 1:], 59) @ np.ones(2)   # P_0(T_0 > 60)
print(f"not returned within 60 steps: {n_unreturned} of {N_PATHS}  (exact P_0(T_0 > 60) = {p_tail:.2e})")
assert n_unreturned == 0, "some path did not return within 60 steps"
ret_hat = mc_estimate(T_ret)
print(f"P(X_40 = 0 | X_0 = 2): {pi0_hat}   exact pi_0 = {pi_mob[0]:.6f}")
print(f"E_0 T_0:               {ret_hat}   exact 1/pi_0 = {ret_exact:.6f}")

n_arr = np.arange(N_STEPS_MOB + 1)
exact_curve = np.array([np.linalg.matrix_power(P_mob, n)[2, 0] for n in n_arr])
fig, ax = plt.subplots()
ax.plot(n_arr, (paths_mob == 0).mean(axis=0), "C0o", ms=4, label=f"MC fraction in state 0 ({N_PATHS} paths, start 2)")
ax.plot(n_arr, exact_curve, "C3-", lw=1.5, label=r"exact $P^n(2,0)$")
ax.axhline(pi_mob[0], color="gray", ls="--", lw=0.8, label=r"$\pi_0 = 22/47$")
ax.set(xlabel="n", ylabel=r"$P(X_n = 0 \mid X_0 = 2)$", xlim=(0, 15),
       title=r"Social mobility chain: convergence at rate $|\lambda_2|^n$, $|\lambda_2| = 0.441$")
ax.legend();

### R3 재유도: 소멸확률 (Extinction Probability)

**명제.** 자손분포 $(p_k)$, pgf $\varphi(s)=\sum_kp_ks^k$, $Z_0=1$, $m=\varphi'(1)$, $p_1<1$. (a) $\varphi_{n+1}=\varphi\circ\varphi_n$ ($\varphi_n(s)=E[s^{Z_n}]$)을 첫 세대(또는 마지막 세대)에 조건화하여 증명하라. (b) 소멸확률 $q=P(Z_n=0\text{ for some }n)$이 $\varphi$의 $[0,1]$에서 **최소** 고정점임을 증명하고, $m\le1$이면 $q=1$임을 볼록성 논법으로 증명하라. (c) $(p_0,p_1,p_2)=(0.25,0.25,0.5)$에서 $q$를 손으로 풀고, $\varphi_{25}(0)$을 코드로 계산하라.

**가정이 왜 필요한가.**
- 개체들의 자손 수가 i.i.d.이고 세대 간 독립: 그래야 $E[s^{Z_{n+1}}\mid Z_n=k]=\varphi(s)^k$ (독립 합의 pgf = 곱).
- $p_1<1$: $p_1=1$이면 $\varphi(s)=s$가 항등이라 모든 점이 고정점이고 $q=0$ (퇴화). 볼록성 논법에서 $\varphi$가 **강**볼록이어야 고정점이 많아야 둘이다.
- 사건의 연속성 $\{Z_n=0\}\uparrow\{\text{소멸}\}$: $q=\lim_n\varphi_n(0)$이 성립하는 이유.

✍️ 내 유도 (R3): (책을 덮고 여기에 (a)(b)(c)를 적으세요)

In [ ]:
# 여기에 풀이: 손으로 푼 q 와 m 을 적어 두고 아래 코드 채점과 비교

<details><summary>기대 답안 (R3)</summary>

**(a) 합성.** $Z_{n+1}=\sum_{j=1}^{Z_n}\xi_j$ ($\xi_j$ i.i.d., $Z_n$과 독립). $Z_n=k$에 조건화하면 $E[s^{Z_{n+1}}\mid Z_n=k]=E[s^{\xi_1}]^k=\varphi(s)^k$, 따라서 $\varphi_{n+1}(s)=E\big[\varphi(s)^{Z_n}\big]=\varphi_n(\varphi(s))$. 즉 $\varphi_{n+1}=\varphi_n\circ\varphi$이고, 귀납으로 $\varphi_n=\varphi^{\circ n}$이므로 $\varphi_{n+1}=\varphi\circ\varphi_n$과 같다. (첫 세대에 조건화해도 같다: $Z_1=k$인 $k$개의 독립 부분과정이 $n$세대 뒤에 모두 $Z^{(j)}_n$ 개체를 가지므로 $\varphi_{n+1}(s)=\sum_kp_k\varphi_n(s)^k=\varphi(\varphi_n(s))$.)

**(b) 최소 고정점.** $\{Z_n=0\}$은 $n$에 대해 증가하는 사건이므로 $q=\lim_nP(Z_n=0)=\lim_n\varphi_n(0)$ (사건의 연속성). $\varphi$는 $[0,1]$에서 연속이므로 $\varphi(q)=\varphi(\lim\varphi_n(0))=\lim\varphi_{n+1}(0)=q$: $q$는 고정점. 임의의 고정점 $s^*\in[0,1]$에 대해 $\varphi$가 증가함수이므로 $0\le s^*\Rightarrow\varphi(0)\le\varphi(s^*)=s^*\Rightarrow\cdots\Rightarrow\varphi_n(0)\le s^*$, 극한에서 $q\le s^*$: 최소. $m\le1$: $\varphi$는 볼록($\varphi''\ge0$)이고 $\varphi(1)=1$, $\varphi'(1)=m\le1$이므로 $s<1$에서 접선 위에 있어 $\varphi(s)\ge1+m(s-1)\ge s$, 그리고 $p_1<1$이면 $s<1$에서 등호가 불가능하여 $\varphi(s)>s$ ($p_k>0$인 $k\ge2$가 있으면 $\varphi$가 강볼록이라 첫 부등호가, 없으면 $\varphi(s)=p_0+p_1s$에서 $m=p_1<1$이라 둘째 부등호가 엄격): 고정점은 $1$뿐, $q=1$. $m>1$이면 $\varphi'(1)>1$이라 $1$ 바로 왼쪽에서 $\varphi(s)<s$, $\varphi(0)=p_0\ge0$이므로 중간값정리로 $[0,1)$에 고정점이 있고 볼록성으로 유일: $q<1$.

**(c) 수치.** $\varphi(s)=\frac14+\frac14s+\frac12s^2$, $m=\frac14+2\cdot\frac12=1.25>1$. $\varphi(s)=s\iff\frac12s^2-\frac34s+\frac14=0\iff2s^2-3s+1=0\iff(2s-1)(s-1)=0$: 근 $\frac12,1$, 최소 근 $q=\frac12$. `np.roots`는 1을 먼저 줄 수도 있다(함정: 최소 근을 골라야 한다). $\varphi_{25}(0)$은 $0\mapsto\varphi(0)\mapsto\cdots$ 25번 반복: $0.4998549$ — $q$와의 차이 $1.5\times10^{-4}$는 $\varphi'(q)^n$ ($\varphi'(1/2)=0.75$) 속도로 줄어든다. (01g 정리 2; Durrett 3e §1.10, Ross 12e §4.7)

</details>

코드 채점 1 — 정확식 vs 정확식: `extinction_probability`(반복)가 손으로 구한 최소 근 $1/2$과 $10^{-8}$ 안에서 일치하는지, 그리고 $\varphi_{25}(0)$을 계산한다. `pgf_iterates`는 01g에서 정의한 도우미라 여기서 다시 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의) — 01g와 동일
def pgf_iterates(pmf, n):
    """[phi_0(0)=0, phi_1(0), ..., phi_n(0)] — 세대 n 까지의 소멸확률."""
    pmf = np.asarray(pmf, dtype=float)
    s, out = 0.0, [0.0]
    for _ in range(n):
        s = float(np.polynomial.polynomial.polyval(s, pmf))
        out.append(s)
    return np.array(out)

pmf_gw = np.array([0.25, 0.25, 0.5])
ks = np.arange(pmf_gw.size)
m_gw = float(pmf_gw @ ks)
q_iter = extinction_probability(pmf_gw)
q_hand = min(r for r in np.roots([pmf_gw[2], pmf_gw[1] - 1, pmf_gw[0]]) if 0 <= r <= 1)   # (c)의 2차방정식 최소 근
phi_gw = pgf_iterates(pmf_gw, 25)
phi25_0 = phi_gw[25]
print(f"m = {m_gw}, q: iteration {q_iter:.12f}, quadratic root {q_hand:.12f};  phi_25(0) = {phi25_0:.7f}")
assert abs(q_iter - 0.5) < 1e-8 and abs(q_hand - 0.5) < 1e-12
print("R3 exact check: ok")

코드 채점 2 — MC vs 정확식: `galton_watson`으로 `N_PATHS`개 개체군을 25세대 돌려 25세대까지의 소멸 비율을 $\varphi_{25}(0)$과 비교한다 ($q$가 아니다 — 함정 9). 그림은 $n=0..25$ 전체에서 소멸 비율이 $\varphi_n(0)$ 곡선을 따라가는 모습. 덤으로 $EZ_{25}=m^{25}$과 중앙값을 같이 출력한다(함정 10).

In [ ]:
N_GEN = 25
Z = galton_watson(pmf_gw, N_GEN, rng, N_PATHS)
ext25_hat = mc_proportion(Z[:, N_GEN] == 0)
ext_curve = (Z == 0).mean(axis=0)
print(f"extinct by gen 25: {ext25_hat}   exact phi_25(0) = {phi25_0:.6f}   (q = {q_iter:.6f})")
print(f"Z_25: mean {Z[:, N_GEN].mean():.1f} (exact m^25 = {m_gw**N_GEN:.1f}), median {np.median(Z[:, N_GEN]):.0f}, "
      f"max {Z[:, N_GEN].max()}")

fig, ax = plt.subplots()
ax.plot(np.arange(N_GEN + 1), phi_gw, "C3-", lw=1.5, label=r"exact $\varphi_n(0)$")
ax.plot(np.arange(N_GEN + 1), ext_curve, "C0o", ms=4, label=f"MC fraction extinct by n ({N_PATHS} paths)")
ax.axhline(q_iter, color="gray", ls="--", lw=0.8, label=f"q = {q_iter:.3f}")
ax.set(xlabel="generation n", ylabel=r"$P(Z_n = 0)$", ylim=(0, 0.6),
       title="Galton-Watson (0.25, 0.25, 0.5): extinction by generation n")
ax.legend(loc="lower right");

## 4. Compare — 재유도 값 · 시뮬레이션 비교

정확값은 모두 3절의 코드가 공식·행렬 계산으로 구한 것이고, 시뮬레이션 열은 3절의 결과다.

"내 예측" 열에는 3절의 ✍️ 칸에 적은 **손 계산** 값을 넣어 보세요 (기본값 `None`).

In [ ]:
hand = {"R1_p_win_045": None, "R1_duration_045": None, "R2_pi_mobility_0": None,
        "R2_return_time_0": None, "R3_ext_by_25": None}   # 손 계산 값을 적어 넣으세요

rows = [
    {"name": "R1_p_win_045",     "predicted": hand["R1_p_win_045"],     "estimate": p_win_hat, "exact": h_exact},
    {"name": "R1_duration_045",  "predicted": hand["R1_duration_045"],  "estimate": dur_hat,   "exact": tau_exact},
    {"name": "R2_pi_mobility_0", "predicted": hand["R2_pi_mobility_0"], "estimate": pi0_hat,   "exact": pi_mob[0]},
    {"name": "R2_return_time_0", "predicted": hand["R2_return_time_0"], "estimate": ret_hat,   "exact": ret_exact},
    {"name": "R3_ext_by_25",     "predicted": hand["R3_ext_by_25"],     "estimate": ext25_hat, "exact": phi25_0},
]
Markdown(compare_table(rows))

각 행이 어긋나면 무엇을 뜻하는가:

- **R1_p_win_045** — 정확식 세 개(유도식·spkit·$NR$)는 이미 $10^{-10}$에서 일치했으므로, 어긋나면 시뮬레이션 쪽: 흡수 상태를 `simulate_chain`이 제대로 고정하는지, 마지막 열 `[:, -1]`을 보고 있는지.
- **R1_duration_045** — 어긋나면 지평 절단(함정 3): 미흡수 경로 수가 0이 아닌데 $T$를 채워 넣었거나, `hitting_times`가 시각 0을 포함하는 관례를 잘못 읽은 것.
- **R2_pi_mobility_0** — 어긋나면 단면을 잘못 잡은 것(시각 40 열이 아니라 시간 평균을 넣었다면 자기상관 때문에 SE가 틀림) 또는 수렴 전 시각을 쓴 것 — 이 사슬은 $|\lambda_2|=0.44$라 10걸음이면 충분.
- **R2_return_time_0** — 어긋나면 귀환시간 관례($T_0\ge1$; 시각 0 제외)나 60걸음 절단. $\pi(x)=1/E_xT_x$ 자체는 정리다.
- **R3_ext_by_25** — 어긋나면 $q$와 $\varphi_{25}(0)$을 혼동(함정 9; 차이 $1.5\times10^{-4}$는 SE $0.0035$보다 작아 이 표본으로는 구별 불가하지만 방향은 항상 아래)하거나 `galton_watson`의 세대 인덱스 오류.

In [ ]:
assert_close(p_win_hat, h_exact, k=4, name="R1_p_win_045")
assert_close(dur_hat, tau_exact, k=4, name="R1_duration_045")
assert_close(pi0_hat, pi_mob[0], k=4, name="R2_pi_mobility_0")
assert_close(ret_hat, ret_exact, k=4, name="R2_return_time_0")
assert_close(ext25_hat, phi25_0, k=4, name="R3_ext_by_25")
print("all MC-vs-exact checks passed (k = 4)")

## 5. 자기채점 (Self-Grading)

각 문항 0/1/2점: 0 = 못 씀, 1 = 핵심은 맞았으나 상수·조건 누락, 2 = 책 없이 정확. 재유도 R1–R3은 유도의 논리(2점 만점)에 코드 채점 통과 여부를 메모로 남긴다 — 정확식 층과 MC 층 중 어느 쪽이 실패했는지까지.

| 항목 | 원 노트북 | 점수 (0–2) | 메모 |
|---|---|---|---|
| Q1 | 01a | | |
| Q2 | 01a | | |
| Q3 | 01b | | |
| Q4 | 01b | | |
| Q5 | 01c | | |
| Q6 | 01c | | |
| Q7 | 01d | | |
| Q8 | 01d | | |
| Q9 | 01e | | |
| Q10 | 01f | | |
| Q11 | 01f | | |
| Q12 | 01g | | |
| R1 파산 재유도 | 01c | | 코드 채점 통과? (정확식 / MC) |
| R2 정상분포 재유도 | 01d | | 코드 채점 통과? (정확식 / MC) |
| R3 소멸확률 재유도 | 01g | | 코드 채점 통과? (정확식 / MC) |

**판정 기준.** 문항 12개 + 재유도 3개 = 15개 항목 × 2점 = **30점** 만점. 24점 이상이면 C1으로 진행. 노트북마다 항목 수가 다르므로 다시 풀기 기준은 **그 노트북 항목 만점의 절반 이하**로 잡는다(아래 표). 기준에 걸린 노트북은 Predict 셀 아래를 지운 사본에서 다시 푼 뒤 진행한다. `log/progress.md`에 총점과 최저 노트북(만점 대비 비율이 가장 낮은 것)을 한 줄로 기록한다.

| 원 노트북 | 항목 | 만점 | 다시 풀기 기준 |
|---|---|---|---|
| 01a | Q1, Q2 | 4 | 2점 이하 |
| 01b | Q3, Q4 | 4 | 2점 이하 |
| 01c | Q5, Q6, R1 | 6 | 3점 이하 |
| 01d | Q7, Q8, R2 | 6 | 3점 이하 |
| 01e | Q9 | 2 | 1점 이하 |
| 01f | Q10, Q11 | 4 | 2점 이하 |
| 01g | Q12, R3 | 4 | 2점 이하 |

**자주 틀리는 곳 (1점의 단골).** 1점을 준 문항이 아래 중 하나에 해당하는지 확인하세요.

- Q3: 기하분포의 성공확률을 $\rho_{xx}$로 씀 (정답 $1-\rho_{xx}$; 방문 횟수 평균 $1/(1-\rho_{xx})$).
- Q6: $\tau(i)$의 부호 — $q>p$이면 $q-p>0$이고 $h(i)$가 작아 $\tau\approx i/(q-p)$에 가깝다; $p=1/2$ 공식 $i(N-i)$를 $p\ne1/2$에 씀.
- Q8: "정상분포가 유일하면 수렴한다"고 답함 (비주기가 빠짐); 세자로 평균을 언급하지 않음.
- Q9: $\lambda_*$에서 $|\lambda_{\min}|$을 빠뜨림; TV 정의의 $\frac12$를 빠뜨림.
- Q11: $\tau$의 상계를 $1/(1-\lambda_2)$로 씀 (정답 $(1+\lambda_2)/(1-\lambda_2)$); 수락률을 효율 척도로 답함.
- Q12: $q$를 "$\varphi(s)=s$의 근"이라고만 답하고 **최소**를 빠뜨림; 임계 감쇠율을 $1/n$ 대신 지수로 답함.
- R2(b): $\mu P=\mu$에서 "사이클을 한 칸 민다"는 핵심 단계 없이 결론만 씀.
- R3(b): $m\le1\Rightarrow q=1$에서 $p_1<1$ 가정을 어디서 썼는지 말하지 못함.

✍️ **내 말로**: 모듈 01 전체를 다섯 문장으로 요약해 `notes/cheatsheet.md`의 '모듈 01' 블록에 적으세요. (힌트: 6절의 첫 항목이 뼈대이지만 그것을 보기 전에 먼저 쓰세요.)

## 6. 정리

- 모듈 01의 뼈대: 정의(마르코프, $P^n$) → 분류(클래스, 재귀/일시, 주기) → 흡수($N,B,t$) → 평형($\pi$) → 수렴($\lambda_*$, 결합) → 설계(세부균형, MH) → 무한 상태(소멸, 재귀).
- 기억해야 할 숫자: 날씨 $\pi=(4/7,3/7)$, $\lambda_2=0.3$; 파산 $0.2683/23.17$; 이동 사슬 $(22,16,9)/47$, $|\lambda_2|=0.441$; Ehrenfest $\text{Bin}(n,1/2)$, 게으른(20) $\lambda_2=0.95$, $t_{\rm mix}=39$; MH $\lambda_2=0.908$, $\tau\approx20$; GW $(0.25,0.25,0.5)$: $q=0.5$, $\varphi_{25}(0)=0.49985$; 3차원 단순 랜덤 워크 귀환확률 $0.3405$.
- 세 재유도(파산·정상분포·소멸)는 모두 "첫 걸음/첫 세대에 조건화"라는 하나의 기법이다 — 차분방정식, 사이클 측도, pgf 합성은 같은 논법의 세 얼굴.
- 함정 10개를 한 장으로: 행렬 거듭제곱, gcd 주기, 지평 절단, 특이 $I-Q$, 주기 vs 수렴, 경험 TV 편향, $\lambda_{\min}$, MCMC SE, $\varphi_n(0)$ vs $q$, 평균 vs 전형.
- 자기채점 24/30 이상이면 C1(PageRank: 01d·01e·01f 재사용)로; 아니면 약한 노트북을 Predict부터 다시.

**Trap 카드.**
Q: 모듈 01에서 "어디서 출발하든 오랜 뒤에는 같아 보인다"가 성립하기 위한 조건 세 개와, 각각이 깨졌을 때 무엇이 남는가?
A: (1) 기약 — 깨지면 출발 클래스별로 극한이 다르다: 흡수확률 $B$로 기술(01c). (2) 비주기 — 깨지면 $P^n$은 진동하고 세자로 평균·시간 평균만 수렴(01d/01e). (3) 유한 또는 양재귀 — 깨지면 영재귀·일시: $p^n(x,y)\to0$이고 정상분포가 없다(01x).

**다음 노트북: C1 (캡스톤 1 — PageRank).** 01d(정상분포 세 계산법), 01e(수렴 속도 $\lambda_2$와 감쇠 계수 $d$), 01f(가역성 여부·거듭제곱 반복)를 한 프로젝트로 묶는다. 복습 점수가 24/30 이상이면 바로 시작; 01d–01f 중 다시 풀기 기준(만점의 절반 이하)에 걸린 노트북이 있으면 그 노트북을 먼저 다시 연다.

`log/progress.md` 한 줄 템플릿: `2026-__-__ | 01r | 총점 __/30 | 최저 노트북: __ (__/__점) | 코드 채점: R1 __ R2 __ R3 __ | 메모: __`